In [5]:
import sys

class SudokuCSP:
    def __init__(self, board):
        self.variables = [(r, c) for r in range(9) for c in range(9)]
        self.domains = {v: list(range(1, 10)) if board[v[0]][v[1]] == 0 
                        else [board[v[0]][v[1]]] for v in self.variables}
        
        self.neighbors = {v: set() for v in self.variables}
        for r, c in self.variables:
            for i in range(9):
                if i != c: self.neighbors[(r, c)].add((r, i))
                if i != r: self.neighbors[(r, c)].add((i, c))
            start_r, start_c = (r // 3) * 3, (c // 3) * 3
            for dr in range(3):
                for dc in range(3):
                    nr, nc = start_r + dr, start_c + dc
                    if (nr, nc) != (r, c):
                        self.neighbors[(r, c)].add((nr, nc))
        
        self.backtrack_calls = 0
        self.backtrack_failures = 0

    def is_consistent(self, var, value, assignment):
        for neighbor in self.neighbors[var]:
            if neighbor in assignment and assignment[neighbor] == value:
                return False
        return True

    def ac3(self):
        queue = []
        for v in self.variables:
            for n in self.neighbors[v]:
                queue.append((v, n))
        while queue:
            (xi, xj) = queue.pop(0)
            if self.revise(xi, xj):
                if not self.domains[xi]: return False
                for xk in self.neighbors[xi]:
                    if xk != xj: queue.append((xk, xi))
        return True

    def revise(self, xi, xj):
        revised = False
        for x in self.domains[xi][:]:
            if not any(x != y for y in self.domains[xj]):
                self.domains[xi].remove(x)
                revised = True
        return revised

    def forward_checking(self, var, value, current_domains):
        new_domains = {v: list(d) for v, d in current_domains.items()}
        for neighbor in self.neighbors[var]:
            if value in new_domains[neighbor]:
                new_domains[neighbor].remove(value)
                if not new_domains[neighbor]: return None
        return new_domains

    def solve(self):
        self.ac3() 
        return self.backtrack({})

    def backtrack(self, assignment):
        self.backtrack_calls += 1
        if len(assignment) == 81: return assignment

        unassigned = [v for v in self.variables if v not in assignment]
        var = min(unassigned, key=lambda v: len(self.domains[v]))

        for value in self.domains[var]:
            if self.is_consistent(var, value, assignment):
                assignment[var] = value
                old_domains = self.domains
                new_domains = self.forward_checking(var, value, self.domains)
                
                if new_domains is not None:
                    self.domains = new_domains
                    result = self.backtrack(assignment)
                    if result: return result
                
                del assignment[var]
                self.domains = old_domains
        self.backtrack_failures += 1
        return None

def load_and_solve(filename):
    try:
        grid = []
        with open(filename, 'r') as f:
            for line in f:
                clean_line = line.strip()
                if not clean_line:
                    continue
                grid.append([int(char) for char in clean_line[:9]])
        
        if len(grid) != 9 or any(len(row) != 9 for row in grid):
            print(f"Error: {filename} does not have a valid 9x9 structure.")
            return

        solver = SudokuCSP(grid)
        solution = solver.solve()
        
        print(f"\n--- Results for {filename} ---")
        if solution:
            for r in range(9):
                print("".join(str(solution[(r, c)]) for c in range(9)))
            print(f"Backtrack Calls: {solver.backtrack_calls}")
            print(f"Backtrack Failures: {solver.backtrack_failures}")
        else:
            print("No solution found.")
            
    except FileNotFoundError:
        print(f"File {filename} not found!")
    except Exception as e:
        print(f"An error occurred while processing {filename}: {e}")

for board_file in ["easy.txt", "medium.txt", "hard.txt", "veryhard.txt"]:
    load_and_solve(board_file)


--- Results for easy.txt ---
784932156
619485327
235176489
578261934
341897562
926543871
453729618
862314795
197658243
Backtrack Calls: 82
Backtrack Failures: 0

--- Results for medium.txt ---
No solution found.

--- Results for hard.txt ---
152346897
437189652
689572314
821637945
543891726
976425183
798253461
365914278
214768539
Backtrack Calls: 112
Backtrack Failures: 30

--- Results for veryhard.txt ---
431867925
652491387
897532164
384976512
519284736
276315849
943728651
765143298
128659473
Backtrack Calls: 654
Backtrack Failures: 572
